<a href="https://colab.research.google.com/github/laoneya31/IAT360-Group-18/blob/main/notebooks/Jasper_Annotation_ImageSize_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1><b>Annotation Review, Image Size and Augmentation</b></h1></center>
<center>IAT 360 Group 18 (Jasper)</center>

**Tasks:** review annotations, class distribution chart and split percentages, tune `imgsz`, run the bonus augmentation comparison.

# 1. Environment

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**. The shared Drive folder must be in your My Drive: right-click `IAT360-Group 18` in *Shared with me* and choose **Organize > Add shortcut**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone our repo and the official YOLO repo. YOLO is pinned to one release so all four members train with identical code.

In [ ]:
import os, sys
REPO = "/content/IAT360-Group-18"
YOLO_TAG = "v8.4.204"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/laoneya31/IAT360-Group-18 {REPO}
# The folder must not be called "ultralytics": /content is on sys.path, so it would hide the real package.
if not os.path.exists("/content/yolo_repo"):
    !git -c advice.detachedHead=false clone -q --depth 1 --branch {YOLO_TAG} https://github.com/ultralytics/ultralytics /content/yolo_repo
%pip install -q /content/yolo_repo kagglehub
%cd {REPO}

Importing required libraries, and checking the software and hardware.

In [ ]:
import glob, hashlib, random, shutil
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

import ultralytics
ultralytics.checks()
from ultralytics import YOLO

sys.path.insert(0, f"{REPO}/src")
import fruit_yolo as fy

fy.check_config()
print("Team Drive folder:", fy.find_team_drive())
print("Baseline settings:", fy.BASELINE)

## 2. Annotation review

The labelling rules are in `docs/LABELLING_RULES.md`. Load the dataset and check it.

In [ ]:
assert fy.fetch_dataset(), "Dataset is not on Drive yet."
report = fy.validate_dataset()
fy.assert_valid(report)

Class distribution and split percentages, for the dataset report and slides.

In [ ]:
counts = pd.DataFrame(report["boxes"]).reindex(fy.CLASS_NAMES).fillna(0).astype(int)
display(counts)
images = pd.Series(report["images"])
print((images / images.sum() * 100).round(1).astype(str) + " %")

ax = counts.plot.barh(figsize=(8, 4), color=["#2a78d6", "#eb6834", "#1baf7a"], width=0.8)
ax.invert_yaxis(); ax.set_xlabel("boxes"); ax.set_title("Boxes per class and split", loc="left")
ax.spines[["top", "right"]].set_visible(False)
(fy.output_root() / "figures").mkdir(parents=True, exist_ok=True)
plt.tight_layout(); plt.savefig(fy.output_root() / "figures" / "class_distribution.png", dpi=160); plt.show()

Example images with labels, for the report.

In [ ]:
import cv2, random
files = random.Random(0).sample(fy.images_in(fy.DATASET / "images" / "test"), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, path in zip(axes, files):
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    for c, x1, y1, x2, y2 in fy.yolo_boxes(fy.DATASET / "labels" / "test" / f"{path.stem}.txt", width, height):
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), max(2, width // 300))
    ax.imshow(image); ax.axis("off"); ax.set_title(path.name, fontsize=8)
plt.tight_layout(); plt.show()

## Hyperparameter tuning

`imgsz`, baseline 640. Each model is evaluated at the size it was trained on. If 800 runs out of GPU memory, tell Daksh.

Only this one value changes. Everything else stays at the shared baseline, so the difference from the baseline can be attributed to it.

In [ ]:
assert fy.fetch_dataset(), "Dataset is not on Drive yet."
OWNER = "Jasper"
PARAM, VALUES = fy.TUNING_PLAN[OWNER]
RUNS = [f"{PARAM}_{v}" for v in VALUES]

for name, value in zip(RUNS, VALUES):
    weights = fy.train_run(name, OWNER, overrides={PARAM: value})
    fy.evaluate(name, weights)

In [ ]:
table = fy.compare_runs(names=["baseline"] + RUNS)
display(table)
fy.plot_comparison(table, f"{PARAM}: mAP@0.5:0.95 on the validation set",
                   save_to=fy.output_root() / "figures" / f"tuning_{PARAM}.png")
plt.show()
for name in RUNS:
    print(name, fy.fit_report(name))

### My results

*TODO: which value won, by how much, and why you think that happened. Copy this into the report.*

Make a zip of your runs for GitHub: unzip it at the top of the repo and commit.

In [ ]:
fy.package_results(RUNS)

## Bonus: augmentation

YOLO's default `hsv_s=0.7` shifts saturation heavily, and colour is the main sign of rot, so the proposal asked for limited
colour jitter. Three conditions: no augmentation, limited, and the baseline's defaults.

In [ ]:
NO_AUGMENT = dict(hsv_h=0, hsv_s=0, hsv_v=0, fliplr=0, scale=0, translate=0, mosaic=0)
LIMITED = dict(hsv_h=0.005, hsv_s=0.2, hsv_v=0.2, fliplr=0.5, scale=0.5)

for name, overrides in {"aug_none": NO_AUGMENT, "aug_limited": LIMITED}.items():
    weights = fy.train_run(name, "Jasper", overrides=overrides)
    fy.evaluate(name, weights)
display(fy.compare_runs(names=["baseline", "aug_none", "aug_limited"]))

### My results

*TODO: did limited colour jitter help the rotten classes in particular?*